# Preprocess TCEQ Galveston wind data for K-means clustering

In [2]:
import pandas as pd

# Define the file periods
periods = ["1998_2006", "2007", "2008_2020", "2020_2024"]

# Loop through each period
for period in periods:
    input_file = f"wind_data_{period}.txt"  # Update with actual file path
    output_file = f"wind_data_extracted_{period}.csv"

    # Read the tab-delimited file
    df = pd.read_csv(input_file, sep="\t", engine="python", dtype=str)

    # Select relevant columns
    df_selected = df[["Year", "Month", "Day", "Start Hour", 
                      "Wind Direction - Resultant (deg) <61104>", 
                      "Wind Speed - Resultant (mph) <61103>"]]

    # Rename columns
    df_selected.columns = ["Year", "Month", "Day", "Hour", "Wind_Direction", "Wind_Speed"]

    # Drop rows with 2020 for the 2008_2020 period
    if period == "2008_2020":
        df_selected = df_selected[df_selected["Year"] != "2020"]

    # Convert time to datetime
    df_selected["Timestamp"] = pd.to_datetime(df_selected[["Year", "Month", "Day", "Hour"]], errors="coerce")

    # Keep only necessary columns
    df_selected = df_selected[["Timestamp", "Wind_Direction", "Wind_Speed"]]

    # Save to CSV
    df_selected.to_csv(output_file, index=False)

    print(f"Extracted data for {period} saved to {output_file}")

print("Data extraction for all periods completed.")


C:\Users\msmillan\AppData\Local\Temp\ipykernel_49440\3049505612.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected["Timestamp"] = pd.to_datetime(df_selected[["Year", "Month", "Day", "Hour"]], errors="coerce")


Extracted data for 1998_2006 saved to wind_data_extracted_1998_2006.csv
Extracted data for 2007 saved to wind_data_extracted_2007.csv


C:\Users\msmillan\AppData\Local\Temp\ipykernel_49440\3049505612.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected["Timestamp"] = pd.to_datetime(df_selected[["Year", "Month", "Day", "Hour"]], errors="coerce")


Extracted data for 2008_2020 saved to wind_data_extracted_2008_2020.csv
Extracted data for 2020_2024 saved to wind_data_extracted_2020_2024.csv
Data extraction for all periods completed.


C:\Users\msmillan\AppData\Local\Temp\ipykernel_49440\3049505612.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected["Timestamp"] = pd.to_datetime(df_selected[["Year", "Month", "Day", "Hour"]], errors="coerce")


In [3]:
import pandas as pd

# Define the list of dataset periods
periods = ["1998_2006", "2007", "2008_2020", "2020_2024"]

# Initialize an empty list to store dataframes
all_data = []

# Loop through each period
for period in periods:
    # Define the input file
    input_file = f"wind_data_extracted_{period}.csv"

    # Read the CSV file for the current period
    df = pd.read_csv(input_file)

    # Append the dataframe to the list
    all_data.append(df)

# Concatenate all the dataframes into one
combined_df = pd.concat(all_data, ignore_index=True)

# Define the output file name
output_file = "combined_wind_data_1998_2024.csv"

# Save the combined data to a CSV file
combined_df.to_csv(output_file, index=False)

print(f"Combined data saved to {output_file}")


Combined data saved to combined_wind_data_1998_2024.csv


In [4]:
import pandas as pd

# Load the combined data
input_file = "combined_wind_data_1998_2024.csv"
df = pd.read_csv(input_file, parse_dates=["Timestamp"])

# Filter for dates between June 1 and August 31 for each year
df_summer = df[
    (df["Timestamp"].dt.month >= 6) & (df["Timestamp"].dt.month <= 8) &
    ~((df["Timestamp"].dt.month == 6) & (df["Timestamp"].dt.day < 1)) &
    ~((df["Timestamp"].dt.month == 8) & (df["Timestamp"].dt.day > 31))
]

# Save the filtered dataset
output_file = "wind_data_summer_Jun01_Aug31_1998_2024.csv"
df_summer.to_csv(output_file, index=False)

print(f"Filtered summer data saved to {output_file}")


Filtered summer data saved to wind_data_summer_Jun01_Aug31_1998_2024.csv


In [8]:
import pandas as pd
import numpy as np

# Load the extracted CSV file
input_csv = "wind_data_summer_Jun01_Aug31_1998_2024.csv"
output_csv = "processed_wind_data_1998_2024.csv"

# Read CSV
df = pd.read_csv(input_csv, parse_dates=["Timestamp"])

# Convert wind speed and direction to numeric (handle missing or malformed values)
df["Wind_Speed"] = pd.to_numeric(df["Wind_Speed"], errors="coerce")
df["Wind_Direction"] = pd.to_numeric(df["Wind_Direction"], errors="coerce")

# Convert wind speed from mph to m/s
df["Wind_Speed"] = df["Wind_Speed"] * 0.44704

# Compute U, V components
df["U"] = -df["Wind_Speed"] * np.sin(np.radians(df["Wind_Direction"]))
df["V"] = -df["Wind_Speed"] * np.cos(np.radians(df["Wind_Direction"]))

# Save to a new CSV file
df.to_csv(output_csv, index=False)

print(f"Processed wind data saved to {output_csv}")


Processed wind data saved to processed_wind_data_1998_2024.csv


In [9]:
import pandas as pd

# Load the processed wind data
input_csv = "processed_wind_data_1998_2024.csv"  # Update with actual file path
output_csv = "daily_avg_wind_1998_2024.csv"

# Read the CSV file
df = pd.read_csv(input_csv, parse_dates=["Timestamp"])

# Extract date and hour from timestamp
df["Date"] = df["Timestamp"].dt.date
df["Hour"] = df["Timestamp"].dt.hour

# Define morning and afternoon periods
morning_mask = df["Hour"].between(3, 7)
afternoon_mask = df["Hour"].between(14, 18)

# Compute daily averages for morning and afternoon periods
daily_avg = df.groupby("Date").agg(
    U_mor=("U", lambda x: x[morning_mask].mean()),
    V_mor=("V", lambda x: x[morning_mask].mean()),
    U_aft=("U", lambda x: x[afternoon_mask].mean()),
    V_aft=("V", lambda x: x[afternoon_mask].mean()),
).reset_index()

# Save to CSV
daily_avg.to_csv(output_csv, index=False)

print(f"Daily averaged wind data saved to {output_csv}")


Daily averaged wind data saved to daily_avg_wind_1998_2024.csv


In [10]:
import pandas as pd
import numpy as np

# Load the processed wind data
input_csv = "processed_wind_data_1998_2024.csv"  # Update if needed
output_csv = "daily_L_S_values_1998_2024.csv"

# Read CSV and parse timestamp
df = pd.read_csv(input_csv, parse_dates=["Timestamp"])

# Ensure U and V columns exist
if "U" not in df.columns or "V" not in df.columns:
    raise ValueError("The CSV file must contain 'U' and 'V' columns.")

# Extract date
df["Date"] = df["Timestamp"].dt.date

# Define functions for L and S
def calculate_L(group):
    return np.sqrt(np.sum(group["U"])**2 + np.sum(group["V"])**2)

def calculate_S(group):
    return np.sum(np.sqrt(group["U"]**2 + group["V"]**2))

# Compute L and S for each day
daily_L_S = df.groupby("Date").apply(lambda g: pd.Series({
    "L": calculate_L(g),
    "S": calculate_S(g)
})).reset_index()

# Save to CSV
daily_L_S.to_csv(output_csv, index=False)

print(f"Daily L and S values saved to {output_csv}")


Daily L and S values saved to daily_L_S_values_1998_2024.csv


C:\Users\msmillan\AppData\Local\Temp\ipykernel_49440\1465661241.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_L_S = df.groupby("Date").apply(lambda g: pd.Series({


In [11]:
import pandas as pd
import numpy as np

# Load the processed wind data
input_csv = "processed_wind_data_1998_2024.csv"  # Update if needed
output_csv = "daily_L_S_Theta_values_1998_2024.csv"

# Read CSV and parse timestamp
df = pd.read_csv(input_csv, parse_dates=["Timestamp"])

# Ensure U and V columns exist
if "U" not in df.columns or "V" not in df.columns:
    raise ValueError("The CSV file must contain 'U' and 'V' columns.")

# Extract date
df["Date"] = df["Timestamp"].dt.date

# Define functions for L, S, and Theta
def calculate_L(group):
    return np.sqrt(np.sum(group["U"])**2 + np.sum(group["V"])**2)

def calculate_S(group):
    return np.sum(np.sqrt(group["U"]**2 + group["V"]**2))

def calculate_theta(U, V):
    if U > 0:
        return (np.pi / 2) - np.arctan(V / U)
    elif U < 0:
        return (3 * np.pi / 2) - np.arctan(V / U)
    elif V > 0 and U == 0:
        return 0
    elif V < 0 and U == 0:
        return np.pi
    else:
        return np.nan  # Undefined case

# Compute L, S, and Theta for each day
daily_L_S_theta = df.groupby("Date").apply(lambda g: pd.Series({
    "L": calculate_L(g),
    "S": calculate_S(g),
    "Theta": calculate_theta(np.sum(g["U"]), np.sum(g["V"]))
})).reset_index()

# Save to CSV
daily_L_S_theta.to_csv(output_csv, index=False)

print(f"Daily L, S, and Theta values saved to {output_csv}")


Daily L, S, and Theta values saved to daily_L_S_Theta_values_1998_2024.csv


C:\Users\msmillan\AppData\Local\Temp\ipykernel_49440\117834009.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  daily_L_S_theta = df.groupby("Date").apply(lambda g: pd.Series({


In [12]:
import pandas as pd
import numpy as np

# Load the existing L, S, and Theta data
input_csv = "daily_L_S_Theta_values_1998_2024.csv"  # Update if needed
output_csv = "daily_L_S_Theta_R_CosSin_values_1998_2024.csv"

# Read CSV
df = pd.read_csv(input_csv)

# Ensure required columns exist
if not all(col in df.columns for col in ["L", "S", "Theta"]):
    raise ValueError("The CSV file must contain 'L', 'S', and 'Theta' columns.")

# Compute R, Cos(Theta), and Sin(Theta)
df["R"] = df["L"] / df["S"]
df["Cos_Theta"] = np.cos(df["Theta"])
df["Sin_Theta"] = np.sin(df["Theta"])

# Save to CSV
df.to_csv(output_csv, index=False)

print(f"Updated daily values with R, Cos(Theta), and Sin(Theta) saved to {output_csv}")


Updated daily values with R, Cos(Theta), and Sin(Theta) saved to daily_L_S_Theta_R_CosSin_values_1998_2024.csv


In [13]:
import pandas as pd

# Input file names
ls_theta_file = "daily_L_S_Theta_R_CosSin_values_1998_2024.csv"  # Contains L, S, Theta, R, Cos(Theta), Sin(Theta)
avg_wind_file = "daily_avg_wind_1998_2024.csv"  # Contains U_mor, U_aft, V_mor, V_aft
output_csv = "combined_daily_wind_data_1998_2024.csv"

# Read both CSV files
df_ls_theta = pd.read_csv(ls_theta_file)
df_avg_wind = pd.read_csv(avg_wind_file)

# Ensure both dataframes have a 'Date' column for merging
if "Date" not in df_ls_theta.columns or "Date" not in df_avg_wind.columns:
    raise ValueError("Both CSV files must contain a 'Date' column for merging.")

# Merge data on 'Date'
merged_df = pd.merge(df_ls_theta, df_avg_wind, on="Date", how="inner")

# Save the merged data to CSV
merged_df.to_csv(output_csv, index=False)

print(f"Combined daily wind data saved to {output_csv}")


Combined daily wind data saved to combined_daily_wind_data_1998_2024.csv


In [14]:
import pandas as pd

# Input file names
ls_theta_file = "daily_L_S_Theta_R_CosSin_values_1998_2024.csv"  # Contains Cos(Theta), Sin(Theta), R
avg_wind_file = "daily_avg_wind_1998_2024.csv"  # Contains U_mor, V_mor, U_aft, V_aft
output_csv = "final_combined_daily_wind_data_1998_2024.csv"

# Read both CSV files
df_ls_theta = pd.read_csv(ls_theta_file)
df_avg_wind = pd.read_csv(avg_wind_file)

# Ensure both dataframes have a 'Date' column for merging
if "Date" not in df_ls_theta.columns or "Date" not in df_avg_wind.columns:
    raise ValueError("Both CSV files must contain a 'Date' column for merging.")

# Select only the required columns
df_ls_theta = df_ls_theta[["Date", "Cos_Theta", "Sin_Theta", "R"]]
df_avg_wind = df_avg_wind[["Date", "U_mor", "V_mor", "U_aft", "V_aft"]]

# Merge data on 'Date'
merged_df = pd.merge(df_avg_wind, df_ls_theta, on="Date", how="inner")

# Save the final selected data to CSV
merged_df.to_csv(output_csv, index=False)

print(f"Final combined daily wind data saved to {output_csv}")


Final combined daily wind data saved to final_combined_daily_wind_data_1998_2024.csv
